# Marine Acoustic Monitoring — Colab
# Tier 2: Use Pretrained Models (Minimal Labels)

BirdNET embeddings — Trained on 6,500+ bird species. Don't use it to identify birds — use it as a feature extractor. The patterns it learned about animal sounds transfer to marine audio. Install via opensoundscape.

Approach:

-Run the humpback detector on all recordings (immediate results, no labels)

-Extract Perch embeddings for 5-second clips across the dataset

-Use UMAP (dimensionality reduction) + HDBSCAN (clustering) to group similar sounding clips

-Listen to examples from each cluster to understand what the groups represent


**Requisito:** Archivo `participant-download.env` (credenciales R2 que dan los organizadores).

## 1. Clonar repo y entrar al directorio

In [51]:
!git clone https://github.com/SALA-AI-LATAM/hackathon-participants.git
%cd hackathon-participants

Cloning into 'hackathon-participants'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 150 (delta 85), reused 113 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 579.59 KiB | 10.54 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/hackathon-participants/hackathon-participants


## 2. Subir credenciales y cargarlas

Cuando ejecutes la celda, Colab te pedirá **subir un archivo**. Elige `participant-download.env`.

In [52]:
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob
import soundfile as sf
import librosa
from scipy.signal import butter, sosfilt
import bioacoustics_model_zoo as bmz
import umap
import hdbscan
import tempfile
import os
from google.colab import files
from pathlib import Path
import tarfile
from pathlib import Path
!pip install -q boto3 tqdm
import r2_download as hd
import sys
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

In [53]:


uploaded = files.upload()  # Abre el diálogo para subir archivo

env_name = "participant-download.env"
if env_name not in uploaded:
    # Por si subiste con otro nombre
    for k in uploaded:
        if "env" in k or k.endswith(".env"):
            env_name = k
            break

if env_name in uploaded:
    Path(env_name).write_bytes(uploaded[env_name])
    for line in Path(env_name).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            key, val = line.replace("export ", "").split("=", 1)
            os.environ[key] = val.strip("'\"")
    print(f"Credenciales cargadas desde {env_name}")
else:
    print("No se subió participant-download.env. Edita la celda y pega las variables o sube el .env y vuelve a ejecutar.")
    os.environ["R2_ENDPOINT"] = "https://YOUR_ACCOUNT.r2.cloudflarestorage.com"
    os.environ["R2_ACCESS_KEY_ID"] = "YOUR_KEY"
    os.environ["R2_SECRET_ACCESS_KEY"] = "YOUR_SECRET"
    os.environ["R2_BUCKET"] = "sala-2026-hackathon-data"

Saving participant-download.env to participant-download.env
Credenciales cargadas desde participant-download.env


## 3. Descargar datos (marine-acoustic-colab)

Los datos se guardan **dentro del repo** en `hackathon_data/` para que el pipeline los encuentre.

In [61]:
# Destino dentro del repo para que pipeline_embeddings.py encuentre los datos
REPO_DIR = Path("/content/hackathon-participants")
DEST_DIR = REPO_DIR / "hackathon_data"
DEST_DIR.mkdir(parents=True, exist_ok=True)

client = hd.get_s3_client()
manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"],
    s3_client=client,
    cache_path=str(REPO_DIR / "manifest.json"),
)
hd.summarize_manifest(manifest)

DATASET = "marine-acoustic-colab"
stats = hd.download_dataset(
    manifest,
    dataset_name=DATASET,
    dest_dir=str(DEST_DIR),
)
print(f"\nDownloaded: {stats['downloaded']}, Skipped: {stats['skipped']}, Failed: {stats['failed']}")

# Extraer el tar.gz del subset colab
tar_path = DEST_DIR / "marine-acoustic-colab" / "marine-acoustic-colab.tar.gz"
if tar_path.exists():
    extract_to = DEST_DIR / "marine-acoustic"
    extract_to.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_to)
    print(f"Extraído en {extract_to}")
    !ls -la "{extract_to}"

Dataset               Shards  Size (GB) Format     Description
--------------------------------------------------------------------------------
precipitation-nowcasting      14       2.38 csv+netcdf+pt Weather station CSVs + LDAS gr
marine-acoustic-colab       1       0.44 tar.gz     Colab-friendly subset of under
marine-acoustic-core     146       7.33 wav        Curated subset of underwater a
bruv-videos               18      69.48 mp4        BRUV underwater video files — 


Shards: 100%|██████████| 1/1 [00:05<00:00,  5.85s/shard, file=marine-acoustic-colab.tar.gz, size_mb=445]
/tmp/ipykernel_296/1303826958.py:28: DeprecationWarning:

Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.




Downloaded: 0, Skipped: 1, Failed: 0
Extraído en /content/hackathon-participants/hackathon_data/marine-acoustic
total 12
drwxr-xr-x 3 root    root  4096 Mar 12 00:23 .
drwxr-xr-x 4 root    root  4096 Mar 12 00:23 ..
drwxr-xr-x 5 1000104 users 4096 Mar  9 19:26 marine-acoustic


## 4. Instalar dependencias del pipeline

In [55]:
# ============================================
# Marine Acoustic Monitoring — Dependencies
# ============================================

# Base (explorer + acoustic_data.py)
!pip install -q \
numpy \
soundfile \
scipy \
matplotlib \
boto3 \
tqdm

# Resampling and playback (notebook listen, model input)
!pip install -q librosa

# Tier 1 — Soundscape indices (scikit-maad)
!pip install -q scikit-maad

# --------------------------------------------
# Optional tiers (uncomment if needed)
# --------------------------------------------

# Tier 2 — Pretrained models (heavy)
!pip install -q opensoundscape
!pip install -q bioacoustics-model-zoo
!pip install -q tensorflow
!pip install -q tensorflow-hub

# Tier 3 — Annotation tools
# !pip install -q whombat

# WhAM (CETI) – whale acoustic model
# !pip install -q git+https://github.com/Project-CETI/wham

In [56]:
!pip install -q opensoundscape==0.12.1 bioacoustics-model-zoo==0.12.0 tensorflow tensorflow-hub

!pip install -q umap-learn hdbscan pyarrow pandas plotly

!pip install -q ai-edge-litert

## 5. Ejecutar pipeline (embeddings + UMAP + HDBSCAN)

In [57]:


# --- Config ---
DATA_DIR = Path("/content/hackathon-participants/hackathon_data/marine-acoustic/marine-acoustic")
OUT_DIR = Path("/content/hackathon-participants/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_CLIPS = 1000
CLIP_DURATION = 5.0
BIRDNET_SR = 35000

# --- 1. Descubrir WAVs ---
wavs = sorted(DATA_DIR.rglob("*.wav"))
print(f"Found {len(wavs)} WAV files")

# --- 2. Segmentar en clips y guardar como WAVs temporales ---
tmp_dir = Path(tempfile.mkdtemp())
clips_meta = []
clip_id = 0

for wav_path in wavs:
    try:
        info = sf.info(str(wav_path))
        sr_orig = info.samplerate
        total_s = info.duration
    except Exception as e:
        print(f"Skip {wav_path.name}: {e}")
        continue

    for start_s in np.arange(0, total_s - CLIP_DURATION, CLIP_DURATION):
        if clip_id >= MAX_CLIPS:
            break
        try:
            audio, sr = sf.read(
                str(wav_path),
                start=int(start_s * sr_orig),
                stop=int((start_s + CLIP_DURATION) * sr_orig),
                dtype='float32'
            )
            if audio.ndim > 1:
                audio = audio.mean(axis=1)

            sos = butter(4, 50, btype='highpass', fs=sr, output='sos')
            audio = sosfilt(sos, audio)

            if sr != BIRDNET_SR:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=BIRDNET_SR)

            clip_path = tmp_dir / f"clip_{clip_id:04d}.wav"
            sf.write(str(clip_path), audio, BIRDNET_SR)

            unit = "pilot"
            for u in ["5783", "6478"]:
                if u in str(wav_path):
                    unit = u
                    break

            clips_meta.append({
                "clip_id": clip_id,
                "file_path": str(wav_path),
                "clip_wav": str(clip_path),
                "start_s": float(start_s),
                "end_s": float(start_s + CLIP_DURATION),
                "unit": unit,
                "source_group": wav_path.parent.name,
            })
            clip_id += 1
        except Exception as e:
            print(f"Error clip {clip_id}: {e}")
            continue
    if clip_id >= MAX_CLIPS:
        break

print(f"Prepared {len(clips_meta)} clips")

# --- 3. Extraer embeddings con BirdNET ---
print("Loading BirdNET model...")
birdnet = bmz.BirdNET()

clip_files = [c["clip_wav"] for c in clips_meta]
print(f"Generating BirdNET embeddings for {len(clip_files)} clips...")
embeddings = birdnet.embed(clip_files, return_dfs=False, batch_size=64, num_workers=0)

emb_array = np.array(embeddings).astype(np.float32)
print(f"Embeddings shape: {emb_array.shape}")

# --- 4. UMAP + HDBSCAN ---
print("Running UMAP + HDBSCAN...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
umap_2d = reducer.fit_transform(emb_array)

clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
labels = clusterer.fit_predict(emb_array)

# --- 5. Guardar resultados ---
clips_df = pd.DataFrame(clips_meta).drop(columns=["clip_wav"])
if len(labels) == len(clips_df):
    clips_df["cluster"] = labels
else:
    clips_df["cluster"] = labels[:len(clips_df)]

clips_df.to_parquet(OUT_DIR / "clips_with_clusters.parquet", index=False)
np.save(OUT_DIR / "umap_2d.npy", umap_2d[:len(clips_df)])
np.save(OUT_DIR / "embeddings_birdnet.npy", emb_array)

print(f"Saved to {OUT_DIR}")
print(f"Clusters found: {len(set(labels) - {-1})}")
print(f"Noise points: {(labels == -1).sum()}")

import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)

Found 11 WAV files
Prepared 1000 clips
Loading BirdNET model...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/opensoundscape/ml/cnn.py:599: UserWarning:


                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT
                    to av

downloading model from URL...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Generating BirdNET embeddings for 1000 clips...


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

Embeddings shape: (1000, 1024)
Running UMAP + HDBSCAN...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Saved to /content/hackathon-participants/outputs
Clusters found: 3
Noise points: 4


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



## 6. Ver resultados: mapa UMAP y escuchar clips

In [58]:


sys.path.insert(0, "/content/hackathon-participants/marine-acoustic-monitoring")
import acoustic_data as ad

out = Path("/content/hackathon-participants/outputs")
clips = pd.read_parquet(out / "clips_with_clusters.parquet")
umap_2d = np.load(out / "umap_2d.npy")

clips["umap_x"] = umap_2d[:, 0]
clips["umap_y"] = umap_2d[:, 1]

fig = px.scatter(
    clips, x="umap_x", y="umap_y", color="cluster",
    hover_data=["clip_id", "unit", "source_group", "start_s", "file_path"],
    height=500,
)
fig.show()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [59]:
!find /content -name "clips_with_clusters.parquet"

/content/hackathon-participants/outputs/clips_with_clusters.parquet


In [60]:
# Escuchar un clip por cluster
from IPython.display import Audio

clip_id = 899  # Cambia para otro clip (revisa clips["clip_id"])
row = clips[clips["clip_id"] == clip_id].iloc[0]
path = row["file_path"]
start_s = float(row["start_s"])
duration_s = float(row["end_s"] - row["start_s"])

audio, sr = ad.load_audio(path, duration_s=duration_s, offset_s=start_s)
if sr > 35000:
    import librosa
    audio = librosa.resample(audio, orig_sr=sr, target_sr=32000)
    sr = 32000
display(Audio(audio, rate=sr))